# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hisham-Walid/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines and verifies the warehouse slice for the CTR / Engagement Opportunity Scoring lane. It uses a mid-panel month and keeps June 2026 sealed.

## 1. Unit of analysis + time window

**Contract answer 1 — unit:** the raw source grain is one `report_date × client_hash_id × content_hash_id` page-day. The modeling frame aggregates that source to **one pseudonymized content page at the March 20 decision cutoff**.

**Contract answer 2 — tables:** this assignment uses only `fact_content_daily_performance/month=2026-03`. The daily fact contains the GSC impressions, clicks, and position signals required for this CTR lane; client and content identifiers are context only.

**Contract answer 3 — windows:** features cover March 1–20, 2026 and are knowable at the end of March 20. The observed proxy covers March 21–31. June 2026 remains a sealed future test month.

The lane slice requires at least 100 feature-window impressions and `gsc_data_available IS TRUE`. A model row is retained for this training demonstration only when it also has an observed outcome-window impression.

In [1]:
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42
FEATURE_START = "2026-03-01"
DECISION_DATE = "2026-03-20"
OUTCOME_START = "2026-03-21"
OUTCOME_END = "2026-03-31"

con = duckdb.connect()
local_file = os.getenv("FLYRANK_MARCH_PARQUET")
hf_token = os.getenv("HF_TOKEN")
if local_file and Path(local_file).is_file():
    escaped = Path(local_file).as_posix().replace("'", "''")
    FACT = f"read_parquet('{escaped}')"
    source_mode = "authenticated local cache"
elif hf_token:
    escaped_token = hf_token.replace("'", "''")
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{escaped_token}')")
    FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
    source_mode = "authenticated Hugging Face stream"
else:
    raise RuntimeError("Set HF_TOKEN or FLYRANK_MARCH_PARQUET; never paste a token into this notebook.")

print(f"Warehouse source ready: March 2026 partition via {source_mode}.")
print(f"Feature window: {FEATURE_START} through {DECISION_DATE}; outcome window: {OUTCOME_START} through {OUTCOME_END}.")


Warehouse source ready: March 2026 partition via authenticated local cache.
Feature window: 2026-03-01 through 2026-03-20; outcome window: 2026-03-21 through 2026-03-31.


## 2. Fields: feature / label / context / excluded

**Contract answer 4 — prediction/proxy:** the ranking target remains future missed-click opportunity. For this short demonstration, the observed outcome proxy is future impressions multiplied by the positive gap between a page's future CTR and the median future CTR of pages in the same position band. A binary `under_capture_label` is used only for the leakage score demonstration; the capstone ranking metric remains NDCG@20.

**Contract answer 5 — deliberate exclusion:** June 2026 and every March 21–31 outcome value are excluded from features because they are future information at the decision moment. I also exclude IDs from model features, all GA4 columns from this GSC-only lane, availability flags after filtering, and product-derived trend fields.

**Output:** a content strategist receives a ranked, capacity-limited queue for human review—not an automatic edit instruction.

In [2]:
feature_contract = pd.DataFrame([
    ("prior_impressions", "feature", "Knowable at the decision moment because it sums March 1–20 GSC impressions."),
    ("prior_clicks", "feature", "Knowable at the decision moment because it sums March 1–20 GSC clicks."),
    ("prior_avg_position", "feature", "Knowable at the decision moment because it is impression-weighted from March 1–20."),
    ("prior_active_days", "feature", "Knowable at the decision moment because it counts observed impression days through March 20."),
    ("prior_ctr_pct", "feature", "Knowable at the decision moment because it uses only March 1–20 clicks and impressions."),
    ("future_missed_clicks", "label / proxy", "Computed only from observed March 21–31 outcomes; never a feature."),
    ("client_hash_id, content_hash_id", "context", "Used for joins, grouped splitting, and grain checks; never learned."),
    ("report_date", "context", "Defines feature and outcome windows; never learned directly."),
    ("June 2026 sample and March 21–31 values", "excluded", "Future information at the March 20 decision moment."),
    ("GA4 fields", "excluded", "Not required for this GSC lane and sparsely available in this month."),
], columns=["field", "bucket", "available when / reason"])

assert (feature_contract["bucket"] == "feature").sum() == 5
display(feature_contract)
print("Feature limit verified: exactly five candidate model features.")


,field,bucket,available when / reason
0,prior_impressions,feature,Knowable at the decision moment because it sum...
1,prior_clicks,feature,Knowable at the decision moment because it sum...
2,prior_avg_position,feature,Knowable at the decision moment because it is ...
3,prior_active_days,feature,Knowable at the decision moment because it cou...
4,prior_ctr_pct,feature,Knowable at the decision moment because it use...
5,future_missed_clicks,label / proxy,Computed only from observed March 21–31 outcom...
6,"client_hash_id, content_hash_id",context,"Used for joins, grouped splitting, and grain c..."
7,report_date,context,Defines feature and outcome windows; never lea...
8,June 2026 sample and March 21–31 values,excluded,Future information at the March 20 decision mo...
9,GA4 fields,excluded,Not required for this GSC lane and sparsely av...


Feature limit verified: exactly five candidate model features.


## 3. Verify it with queries (grain, counts, availability)

The next cell runs **exactly three numbered verification queries** against the March partition: raw grain, count/date span, and availability. The feature-frame build follows as a separate construction query. Availability uses `IS TRUE`; NULL and FALSE are therefore both excluded rather than misread as zero activity.

In [3]:
verification_queries = {
    "1 — raw grain": f"""
        SELECT COUNT(*) AS duplicate_page_day_groups
        FROM (
            SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
            FROM {FACT}
            GROUP BY 1, 2, 3
            HAVING COUNT(*) > 1
        )
    """,
    "2 — count and date span": f"""
        SELECT COUNT(*) AS row_count,
               COUNT(DISTINCT client_hash_id) AS clients,
               COUNT(DISTINCT content_hash_id) AS content_pages,
               MIN(report_date) AS min_date,
               MAX(report_date) AS max_date
        FROM {FACT}
    """,
    "3 — availability with IS TRUE": f"""
        SELECT COUNT(*) AS total_rows,
               COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true_rows,
               ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS gsc_true_pct,
               COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
               ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS ga4_true_pct
        FROM {FACT}
    """,
}

verification_results = {}
for name, query in verification_queries.items():
    result = con.execute(query).df()
    verification_results[name] = result
    print(name)
    display(result)

assert verification_results["1 — raw grain"].loc[0, "duplicate_page_day_groups"] == 0
assert str(verification_results["2 — count and date span"].loc[0, "min_date"])[:10] == FEATURE_START
assert str(verification_results["2 — count and date span"].loc[0, "max_date"])[:10] == OUTCOME_END

feature_query = f"""
WITH daily AS (
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_sum_position
    FROM {FACT}
    WHERE gsc_data_available IS TRUE
),
prior AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS prior_impressions,
           SUM(gsc_clicks) AS prior_clicks,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS prior_avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS prior_active_days,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS prior_ctr_pct
    FROM daily
    WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
outcome AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS future_impressions,
           SUM(gsc_clicks) AS future_clicks,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS future_avg_position,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS future_ctr_pct
    FROM daily
    WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) > 0
)
SELECT p.*, o.future_impressions, o.future_clicks, o.future_avg_position, o.future_ctr_pct
FROM prior p
JOIN outcome o USING (client_hash_id, content_hash_id)
"""

model_frame = con.execute(feature_query).df()
feature_columns = ["prior_impressions", "prior_clicks", "prior_avg_position", "prior_active_days", "prior_ctr_pct"]
assert model_frame[["client_hash_id", "content_hash_id"]].duplicated().sum() == 0
assert len(feature_columns) == 5

position_bins = [-np.inf, 3, 10, 20, 50, np.inf]
position_labels = ["top_3", "page_1", "striking", "page_3_5", "deep"]
model_frame["future_position_band"] = pd.cut(model_frame["future_avg_position"], bins=position_bins, labels=position_labels)
model_frame["peer_expected_future_ctr_pct"] = model_frame.groupby("future_position_band", observed=True)["future_ctr_pct"].transform("median")
model_frame["future_ctr_gap_pp"] = (model_frame["peer_expected_future_ctr_pct"] - model_frame["future_ctr_pct"]).clip(lower=0)
model_frame["future_missed_clicks"] = model_frame["future_impressions"] * model_frame["future_ctr_gap_pp"] / 100
model_frame["under_capture_label"] = (model_frame["future_missed_clicks"] > 0).astype(int)

public_preview = model_frame[feature_columns + ["future_missed_clicks"]].head(8)
display(public_preview.style.format({"prior_avg_position": "{:.2f}", "prior_ctr_pct": "{:.3f}", "future_missed_clicks": "{:.2f}"}))
print(f"Feature frame: {len(model_frame):,} unique pages across {model_frame['client_hash_id'].nunique()} pseudonymized clients.")
print(f"Observed under-capture proxy base rate: {model_frame['under_capture_label'].mean():.1%}.")


1 — raw grain


,duplicate_page_day_groups
0,0


2 — count and date span


,row_count,clients,content_pages,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


3 — availability with IS TRUE


,total_rows,gsc_true_rows,gsc_true_pct,ga4_true_rows,ga4_true_pct
0,9841378,3611061,36.7,413966,4.2


,prior_impressions,prior_clicks,prior_avg_position,prior_active_days,prior_ctr_pct,future_missed_clicks
0,795.000000,1.000000,3.61,20,0.126,0.00
1,461.000000,0.000000,2.19,20,0.000,0.07
2,3157.000000,2.000000,5.45,20,0.063,0.00
3,1181.000000,2.000000,4.35,20,0.169,0.00
4,8568.000000,31.000000,2.38,20,0.362,0.00
5,375.000000,0.000000,5.31,20,0.000,0.00
6,6299.000000,22.000000,3.52,20,0.349,0.08
7,168.000000,0.000000,27.49,20,0.000,0.00


Feature frame: 86,876 unique pages across 39 pseudonymized clients.
Observed under-capture proxy base rate: 28.7%.


## 4. Leakage trap + data limits

For the required trap, I first train a small client-held-out classifier using only the five pre-decision features. Then I add one forbidden column, `leaky_target_copy`, which is copied directly from the outcome-derived label. Its score should jump to perfect or nearly perfect for the wrong reason. The code deletes the column immediately afterward and asserts that the retained frame and feature list are honest. This ROC-AUC is a leakage lesson, not the capstone's ranking result.

**Named limitation:** this is one mid-panel month with an unbalanced client panel. Only clients with sufficient GSC activity in both windows enter the demonstration, so the observed score may not generalize to new clients, sparse pages, seasonality, or other months. GA4 availability is especially limited here. The data is observational and cannot show that an edit causes more clicks.

In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(splitter.split(model_frame, groups=model_frame["client_hash_id"]))

def quick_model(columns):
    model = RandomForestClassifier(
        n_estimators=100, max_depth=6, min_samples_leaf=50,
        class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
    )
    model.fit(model_frame.iloc[train_idx][columns], model_frame.iloc[train_idx]["under_capture_label"])
    probabilities = model.predict_proba(model_frame.iloc[test_idx][columns])[:, 1]
    return roc_auc_score(model_frame.iloc[test_idx]["under_capture_label"], probabilities)

honest_auc = quick_model(feature_columns)
model_frame["leaky_target_copy"] = model_frame["under_capture_label"]
leaky_auc = quick_model(feature_columns + ["leaky_target_copy"])
model_frame.drop(columns=["leaky_target_copy"], inplace=True)

assert "leaky_target_copy" not in model_frame.columns
assert "leaky_target_copy" not in feature_columns
assert set(feature_columns).isdisjoint({"future_missed_clicks", "under_capture_label", "client_hash_id", "content_hash_id"})
print(f"Honest five-feature held-out ROC-AUC: {honest_auc:.3f}")
print(f"With one copied-label leak ROC-AUC: {leaky_auc:.3f}")
print("Leak removed. The retained model frame contains no copied-label feature.")
print(f"Held-out clients: {model_frame.iloc[test_idx]['client_hash_id'].nunique()}; retained honest features: {len(feature_columns)}.")


Honest five-feature held-out ROC-AUC: 0.781
With one copied-label leak ROC-AUC: 1.000
Leak removed. The retained model frame contains no copied-label feature.
Held-out clients: 10; retained honest features: 5.


## Self-check

Before submitting, I verified:

- [x] Five plain-language contract answers are present
- [x] Exactly three verification queries have visible outputs, including `IS TRUE` availability
- [x] The feature frame contains exactly five pre-decision features with an availability explanation for each
- [x] The deliberate label-derived leak is demonstrated, deleted, and excluded from the honest feature list
- [x] At least one limitation is named and claims remain observational and decision-support only
- [x] The notebook runs top to bottom with no errors and displays no client identifiers, URLs, private queries, or secrets
- [x] The completed notebook is under `work/notebooks/` and ready to commit